# Simulation: Impact of Digby Exponent and Variance Factor Fixes

This notebook compares METACARPA's p-value meta-analysis under four configurations:

| Label | Digby exponent | Factor of 2 in variance | Description |
|-------|---------------|------------------------|-------------|
| **Old** | 0.75 | No | Original METACARPA |
| **Fix exponent only** | pi/4 | No | Only the exponent fix |
| **Fix variance only** | 0.75 | Yes | Only the factor-of-2 fix |
| **New (both fixes)** | pi/4 | Yes | Current METACARPA |

We simulate two GWAS under the null (no true effect) with correlated z-scores
(representing sample overlap), then run the Province & Borecki Stouffer meta-analysis
and measure type 1 error and genomic inflation.

In [ ]:
library(data.table)
library(MASS)
set.seed(42)

## Helper Functions

These replicate the METACARPA meta-analysis pipeline in R with configurable parameters.

In [ ]:
# Digby's tetrachoric correlation from a 2x2 table
digby_rho <- function(n00, n01, n10, n11, exponent = pi/4) {
  if (n01 == 0 || n10 == 0) return(NA_real_)
  alpha <- (n00 * n11) / (n01 * n10)
  ap <- alpha^exponent
  (ap - 1) / (ap + 1)
}

# Estimate tetrachoric rho from two vectors of binarised p-values
estimate_rho <- function(bpval1, bpval2, exponent = pi/4) {
  tbl <- table(factor(bpval1, levels = c(0, 1)),
               factor(bpval2, levels = c(0, 1)))
  digby_rho(tbl[1,1], tbl[1,2], tbl[2,1], tbl[2,2], exponent = exponent)
}

# Off-diagonal cross-product (upper triangle), with optional factor of 2
off_diag_product <- function(rho_mat, w, use_factor2 = TRUE) {
  K <- length(w)
  s <- 0
  for (i in 1:(K-1)) {
    for (j in (i+1):K) {
      s <- s + w[i] * w[j] * rho_mat[i, j]
    }
  }
  if (use_factor2) 2 * s else s
}

# Province & Borecki corrected Stouffer meta-analysis (p-value method)
# Returns corrected p-values for each variant
meta_analyse_stouffer <- function(z1, z2, n1, n2, rho_est,
                                  use_factor2 = TRUE) {
  # Weights: sqrt(N_k) / sqrt(sum(N))
  w <- sqrt(c(n1, n2)) / sqrt(n1 + n2)
  
  # Combined z-score
  z_meta <- w[1] * z1 + w[2] * z2
  
  # Variance of combined z under Province & Borecki:
  # Var = sum(w_k^2) + 2 * sum_{k<l} w_k w_l rho_kl
  # Since sum(w_k^2) = 1 by construction, this simplifies to:
  # Var = 1 + 2 * w1 * w2 * rho
  rho_mat <- matrix(c(1, rho_est, rho_est, 1), 2, 2)
  zse <- sqrt(1 + off_diag_product(rho_mat, w, use_factor2 = use_factor2))
  
  # Corrected p-value
  2 * pnorm(-abs(z_meta / zse))
}

lambda_gc <- function(p) {
  p <- as.numeric(p)
  p <- p[!is.na(p) & p > 0 & p < 1]
  median(qnorm(p/2)^2) / qchisq(0.5, 1)
}

## Simulation

In [ ]:
n_variants <- 100000
n1 <- 10000
n2 <- 10000
rho_values <- c(0.0, 0.1, 0.2, 0.3, 0.5, 0.7)
n_reps <- 5

configs <- data.table(
  label     = c("Old (0.75, no 2x)",
                "Fix exponent only",
                "Fix variance only",
                "New (pi/4 + 2x)"),
  exponent  = c(0.75,  pi/4, 0.75,  pi/4),
  factor2   = c(FALSE, FALSE, TRUE,  TRUE)
)

results <- data.table()

for (rho_true in rho_values) {
  for (rep in 1:n_reps) {
    cat(sprintf("rho=%.1f rep=%d\n", rho_true, rep))
    
    # Simulate correlated z-scores under the null
    Sigma <- matrix(c(1, rho_true, rho_true, 1), 2, 2)
    Z <- mvrnorm(n_variants, mu = c(0, 0), Sigma = Sigma)
    z1 <- Z[, 1]
    z2 <- Z[, 2]
    
    # Convert to p-values and betas (SE = 1/sqrt(N))
    p1 <- 2 * pnorm(-abs(z1))
    p2 <- 2 * pnorm(-abs(z2))
    beta1 <- z1 / sqrt(n1)
    beta2 <- z2 / sqrt(n2)
    
    # Binarise p-values using sign of beta (the --use-beta-sign method)
    bpval1 <- as.integer(qnorm(p1/2) * sign(beta1) <= 0)
    bpval2 <- as.integer(qnorm(p2/2) * sign(beta2) <= 0)
    
    # Uncorrected Stouffer (no overlap correction)
    w <- sqrt(c(n1, n2)) / sqrt(n1 + n2)
    z_uncorrected <- w[1] * z1 + w[2] * z2
    p_uncorrected <- 2 * pnorm(-abs(z_uncorrected))
    
    for (ci in 1:nrow(configs)) {
      cfg <- configs[ci]
      
      # Estimate rho with this exponent
      rho_est <- estimate_rho(bpval1, bpval2, exponent = cfg$exponent)
      
      # Meta-analyse with this variance formula
      p_corrected <- meta_analyse_stouffer(
        z1, z2, n1, n2, rho_est,
        use_factor2 = cfg$factor2
      )
      
      results <- rbind(results, data.table(
        rho_true  = rho_true,
        rep       = rep,
        config    = cfg$label,
        exponent  = cfg$exponent,
        factor2   = cfg$factor2,
        rho_est   = rho_est,
        lambda_gc = lambda_gc(p_corrected),
        t1e_05    = mean(p_corrected < 0.05, na.rm = TRUE),
        t1e_01    = mean(p_corrected < 0.01, na.rm = TRUE),
        t1e_001   = mean(p_corrected < 0.001, na.rm = TRUE)
      ))
    }
    
    # Also store uncorrected
    results <- rbind(results, data.table(
      rho_true  = rho_true,
      rep       = rep,
      config    = "Uncorrected",
      exponent  = NA_real_,
      factor2   = NA,
      rho_est   = NA_real_,
      lambda_gc = lambda_gc(p_uncorrected),
      t1e_05    = mean(p_uncorrected < 0.05, na.rm = TRUE),
      t1e_01    = mean(p_uncorrected < 0.01, na.rm = TRUE),
      t1e_001   = mean(p_uncorrected < 0.001, na.rm = TRUE)
    ))
  }
}

cat("Done.\n")

## Summary Statistics

In [ ]:
summ <- results[, .(
  rho_est   = mean(rho_est, na.rm = TRUE),
  lambda_gc = mean(lambda_gc),
  t1e_05    = mean(t1e_05),
  t1e_01    = mean(t1e_01),
  t1e_001   = mean(t1e_001)
), by = .(rho_true, config)]

# Order configs nicely
summ[, config := factor(config, levels = c(
  "Old (0.75, no 2x)",
  "Fix exponent only",
  "Fix variance only",
  "New (pi/4 + 2x)",
  "Uncorrected"
))]
setorder(summ, rho_true, config)

cat("=== Estimated rho ===")
dcast(summ, rho_true ~ config, value.var = "rho_est")

In [ ]:
cat("=== Lambda GC ===")
dcast(summ, rho_true ~ config, value.var = "lambda_gc")

In [ ]:
cat("=== Type 1 Error (alpha = 0.05) ===")
dcast(summ, rho_true ~ config, value.var = "t1e_05")

In [ ]:
cat("=== Type 1 Error (alpha = 0.001) ===")
dcast(summ, rho_true ~ config, value.var = "t1e_001")

## Plots

In [ ]:
# Colour palette
cols <- c(
  "Old (0.75, no 2x)"  = "#D55E00",
  "Fix exponent only"  = "#E69F00",
  "Fix variance only"  = "#56B4E9",
  "New (pi/4 + 2x)"    = "#009E73",
  "Uncorrected"        = "grey60"
)
pchs <- c(19, 17, 15, 18, 4)
config_levels <- names(cols)

In [ ]:
# Plot 1: Estimated rho vs true rho
options(repr.plot.width = 8, repr.plot.height = 6)
par(mar = c(5, 5, 4, 2))

plot(NA, xlim = c(0, 0.75), ylim = c(-0.05, 0.75),
     xlab = expression("True " * rho),
     ylab = expression("Estimated " * rho),
     main = "Tetrachoric Correlation Recovery",
     cex.main = 1.3, cex.lab = 1.2)
abline(0, 1, col = "grey40", lty = 2, lwd = 2)

for (i in 1:4) {
  cfg <- config_levels[i]
  d <- summ[config == cfg]
  lines(d$rho_true, d$rho_est, col = cols[cfg], lwd = 2)
  points(d$rho_true, d$rho_est, col = cols[cfg], pch = pchs[i], cex = 1.5)
}

legend("topleft", legend = config_levels[1:4],
       col = cols[1:4], pch = pchs[1:4], lwd = 2,
       cex = 0.9, bg = "white")

In [ ]:
# Plot 2: Lambda GC
options(repr.plot.width = 8, repr.plot.height = 6)
par(mar = c(5, 5, 4, 2))

yr <- range(summ$lambda_gc, na.rm = TRUE)
plot(NA, xlim = c(0, 0.75), ylim = yr,
     xlab = expression("True " * rho),
     ylab = expression("Lambda"[GC]),
     main = "Genomic Inflation Factor",
     cex.main = 1.3, cex.lab = 1.2)
abline(h = 1, col = "grey40", lty = 2, lwd = 2)

for (i in seq_along(config_levels)) {
  cfg <- config_levels[i]
  d <- summ[config == cfg]
  lines(d$rho_true, d$lambda_gc, col = cols[cfg], lwd = 2)
  points(d$rho_true, d$lambda_gc, col = cols[cfg], pch = pchs[i], cex = 1.5)
}

legend("topleft", legend = config_levels,
       col = cols, pch = pchs, lwd = 2,
       cex = 0.9, bg = "white")

In [ ]:
# Plot 3: Type 1 error at alpha = 0.05
options(repr.plot.width = 8, repr.plot.height = 6)
par(mar = c(5, 5, 4, 2))

yr <- range(summ$t1e_05, na.rm = TRUE)
plot(NA, xlim = c(0, 0.75), ylim = yr,
     xlab = expression("True " * rho),
     ylab = "Type 1 Error Rate",
     main = expression("Type 1 Error (" * alpha * " = 0.05)"),
     cex.main = 1.3, cex.lab = 1.2)
abline(h = 0.05, col = "grey40", lty = 2, lwd = 2)

for (i in seq_along(config_levels)) {
  cfg <- config_levels[i]
  d <- summ[config == cfg]
  lines(d$rho_true, d$t1e_05, col = cols[cfg], lwd = 2)
  points(d$rho_true, d$t1e_05, col = cols[cfg], pch = pchs[i], cex = 1.5)
}

legend("topleft", legend = config_levels,
       col = cols, pch = pchs, lwd = 2,
       cex = 0.9, bg = "white")

In [ ]:
# Plot 4: Type 1 error at alpha = 0.001
options(repr.plot.width = 8, repr.plot.height = 6)
par(mar = c(5, 5, 4, 2))

yr <- range(summ$t1e_001, na.rm = TRUE)
plot(NA, xlim = c(0, 0.75), ylim = yr,
     xlab = expression("True " * rho),
     ylab = "Type 1 Error Rate",
     main = expression("Type 1 Error (" * alpha * " = 0.001)"),
     cex.main = 1.3, cex.lab = 1.2)
abline(h = 0.001, col = "grey40", lty = 2, lwd = 2)

for (i in seq_along(config_levels)) {
  cfg <- config_levels[i]
  d <- summ[config == cfg]
  lines(d$rho_true, d$t1e_001, col = cols[cfg], lwd = 2)
  points(d$rho_true, d$t1e_001, col = cols[cfg], pch = pchs[i], cex = 1.5)
}

legend("topleft", legend = config_levels,
       col = cols, pch = pchs, lwd = 2,
       cex = 0.9, bg = "white")

## Conclusions

In [ ]:
cat("=" |> rep(70) |> paste(collapse = ""), "\n")
cat("SUMMARY\n")
cat("=" |> rep(70) |> paste(collapse = ""), "\n\n")

# Compare at highest overlap
for (rho in c(0.3, 0.5, 0.7)) {
  cat(sprintf("At rho = %.1f:\n", rho))
  s <- summ[rho_true == rho]
  for (cfg in config_levels) {
    row <- s[config == cfg]
    if (nrow(row) > 0) {
      cat(sprintf("  %-25s  rho_est=%.4f  lambda=%.4f  T1E(0.05)=%.4f  T1E(0.001)=%.5f\n",
                  cfg, row$rho_est, row$lambda_gc, row$t1e_05, row$t1e_001))
    }
  }
  cat("\n")
}